In [2]:
from TableToDict import GetDFv2, GetDF, DisplayColumns
from HuRI import GetHuRI
#from ccsblib import  huri
from Load_CCSB_Data import GetOrfMapping
from load_edgotyping_data import load_qY2H
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns

from PandasToMySQL import MakeSummaryTable

%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.max_rows = 1000
pd.options.display.max_columns = 60

In [3]:
allele_info_df = pd.read_csv('../database_management/20220228_Mikko_table_creation_and_updates/VUS_wt_allele_collection_entry_plates_allele_infos.csv', index_col=0)
#allele_info_df.query('orf_id == 7215.0 and mutation_id == 1976.0')
allele_info_d = {(orf_id,mutation_id):(nt_change,aa_change,'to_be_verified') for orf_id,mutation_id,nt_change,aa_change 
                 in zip(allele_info_df.orf_id, allele_info_df.mutation_id, allele_info_df.ensg, allele_info_df.nt_change)
                 if mutation_id == mutation_id
                }

allele_collection = GetDFv2()

for i, (orf_id,nt_change,aa_change,symbol,ensg,entrez,rc4_mut_id,cegs_mut_id,vus_mut_id,rc4_plate,rc4_well,cegs_plate,cegs_well,vus_plate,vus_well,chr,chr_pos_37,chr_pos_38,cv_allele_id,hgmd_id,cv_clinical_significance,cv_phenotypes,cv_phenotype_ids,hgmd_clinical_significance,hgmd_phenotype,clinical_significance) in allele_collection.iterrows():
    for mut_id in [rc4_mut_id,cegs_mut_id,vus_mut_id]:
        if mut_id == mut_id:
            allele_info_d[orf_id,mut_id] = (nt_change,aa_change,clinical_significance)
            
orf_mapping = GetOrfMapping(orf_ids=allele_info_df.orf_id.unique().tolist())
orf_to_symbol = dict(zip(orf_mapping.orf_id, orf_mapping.symbol))


pdest_layout_df = pd.read_csv('../experiment_designs/20220513_cp_pdest_dual_VUS/VUS_PDEST_dual_experiment_table_annotated.csv', index_col=0)
col = ['exp_plate', 'exp_well', 'orf_id', 'mut_id', 'node_type']
pdest_layout_df = pdest_layout_df[col]
pdest_layout_df['dest_pla_id'] = pdest_layout_df.exp_plate.apply(lambda x: int(x.split('_')[-1]))
pdest_layout_df.head()

pdest_layout_b5_df = pd.read_csv('../experiment_designs/20220803_pDEST_DUAL_batch5/cp_files_used/summary_CP_VUSDUALXP_batch5.csv', index_col=0)

col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']
pdest_layout_b5_df = pdest_layout_b5_df[col]
pdest_layout_b5_df.rename(columns = {'dest_pla':'exp_plate', 'dest_well':'exp_well'}, inplace=True)

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_df])
pdest_layout_df['nt_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][0] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
pdest_layout_df['aa_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][1] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
pdest_layout_df['clinical_significance'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][2] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

pdest_layout_df['symbol'] = pdest_layout_df.orf_id.map(orf_to_symbol)

pdest_layout_df.head()

,orf_id,mut_id,node_type,dest_pla_id,exp_plate,exp_well,nt_change,aa_change,clinical_significance,symbol
78,56360.0,0.0,wt,23,VUSDUALXP_23,A12,WT,WT,WT,NEXN
79,56360.0,0.0,wt,23,VUSDUALXP_23,B12,WT,WT,WT,NEXN
80,56360.0,0.0,wt,23,VUSDUALXP_23,C12,WT,WT,WT,NEXN
81,56360.0,0.0,wt,23,VUSDUALXP_23,D12,WT,WT,WT,NEXN
10,9905.0,0.0,wt,23,VUSDUALXP_23,F02,WT,WT,WT,PNKP


In [7]:
pdest_layout_df.to_csv('pdest_experimental_layout.csv', index=None)

In [5]:
pdest_layout_df.head()

,orf_id,mut_id,node_type,dest_pla_id,exp_plate,exp_well,nt_change,aa_change,clinical_significance,symbol
78,56360.0,0.0,wt,23,VUSDUALXP_23,A12,WT,WT,WT,NEXN
79,56360.0,0.0,wt,23,VUSDUALXP_23,B12,WT,WT,WT,NEXN
80,56360.0,0.0,wt,23,VUSDUALXP_23,C12,WT,WT,WT,NEXN
81,56360.0,0.0,wt,23,VUSDUALXP_23,D12,WT,WT,WT,NEXN
10,9905.0,0.0,wt,23,VUSDUALXP_23,F02,WT,WT,WT,PNKP
